# 数据读取

In [8]:
import torch

test_path = "/home/charles/HZU/Data_processed/HSML/KAIST/current&temp_raw_wl_5000/test/pyg/graph.pt"
train_path = "/home/charles/HZU/Data_processed/HSML/KAIST/current&temp_raw_wl_5000/train/pyg/train_graph_with_labelmask.pt"
val_path   = "/home/charles/HZU/Data_processed/HSML/KAIST/current&temp_raw_wl_5000/val/pyg/graph.pt"

test_graph  = torch.load(test_path, map_location="cpu")
train_graph = torch.load(train_path, map_location="cpu")
val_graph   = torch.load(val_path, map_location="cpu")

def inspect_graph(g, name):
    print(f"\n===== {name} =====")
    print(g)
    print("num_nodes:", g.num_nodes)
    print("num_edges:", g.edge_index.size(1))
    print("node features x:", None if g.x is None else g.x.shape)
    print("edge_index:", g.edge_index.shape)
    if hasattr(g, "y"):
        print("labels y:", g.y.shape)
    if hasattr(g, "train_mask"):
        print("train_mask:", g.train_mask.shape)
    if hasattr(g, "val_mask"):
        print("val_mask:", g.val_mask.shape)
    if hasattr(g, "test_mask"):
        print("test_mask:", g.test_mask.shape)

inspect_graph(train_graph, "TRAIN")
inspect_graph(val_graph, "VAL")
inspect_graph(test_graph, "TEST")



===== TRAIN =====
Data(x=[45000, 5], edge_index=[2, 900000], y=[45000], train_withlabel_mask=[45000], train_nolabel_mask=[45000])
num_nodes: 45000
num_edges: 900000
node features x: torch.Size([45000, 5])
edge_index: torch.Size([2, 900000])
labels y: torch.Size([45000])

===== VAL =====
Data(x=[7500, 5], edge_index=[2, 150000], y=[7500])
num_nodes: 7500
num_edges: 150000
node features x: torch.Size([7500, 5])
edge_index: torch.Size([2, 150000])
labels y: torch.Size([7500])

===== TEST =====
Data(x=[22500, 5], edge_index=[2, 450000], y=[22500])
num_nodes: 22500
num_edges: 450000
node features x: torch.Size([22500, 5])
edge_index: torch.Size([2, 450000])
labels y: torch.Size([22500])


/tmp/ipykernel_30796/1019197295.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test_graph  = torch.load(test_path, map_location="cpu")
/tmp/ipykernel_30796/1019197295.p

## 数据选择，BPFO数据只有一路电流，先用全部有的数据进行分类，用于后期缺失值补全后的效果做对比

In [9]:
import torch

def check_nan_inf(x):
    return {
        "has_nan": torch.isnan(x).any().item(),
        "has_inf": torch.isinf(x).any().item()
    }

def process_x(data, name):
    print(f"\n===== {name} =====")

    assert data.x is not None, f"{name}: data.x is None"

    # ================= 1️⃣ 删除前检查 =================
    print("Original data.x shape:", data.x.shape)
    before_status = check_nan_inf(data.x)
    print("Before drop - Has NaN:", before_status["has_nan"])
    print("Before drop - Has Inf:", before_status["has_inf"])

    # ================= 2️⃣ 删除最后两列 =================
    if data.x.size(1) < 3:
        raise ValueError(f"{name}: data.x has less than 3 features, cannot drop last two columns")

    data.x = data.x[:, :-2]

    # ================= 3️⃣ 删除后检查 =================
    print("After drop shape:", data.x.shape)
    after_status = check_nan_inf(data.x)
    print("After drop - Has NaN:", after_status["has_nan"])
    print("After drop - Has Inf:", after_status["has_inf"])

    # ================= 4️⃣ 结论提示 =================
    if before_status["has_nan"] or before_status["has_inf"]:
        print("⚠️ WARNING: Invalid values already exist BEFORE feature drop")

    if after_status["has_nan"] or after_status["has_inf"]:
        print("⚠️ WARNING: Invalid values remain AFTER feature drop")

    if not any(before_status.values()) and not any(after_status.values()):
        print("✅ data.x is clean before and after drop")

# ===== 执行 =====
process_x(train_graph, "TRAIN")
process_x(val_graph,   "VAL")
process_x(test_graph,  "TEST")



===== TRAIN =====
Original data.x shape: torch.Size([45000, 5])
Before drop - Has NaN: True
Before drop - Has Inf: False
After drop shape: torch.Size([45000, 3])
After drop - Has NaN: False
After drop - Has Inf: False
⚠️ WARNING: Invalid values already exist BEFORE feature drop

===== VAL =====
Original data.x shape: torch.Size([7500, 5])
Before drop - Has NaN: True
Before drop - Has Inf: False
After drop shape: torch.Size([7500, 3])
After drop - Has NaN: False
After drop - Has Inf: False
⚠️ WARNING: Invalid values already exist BEFORE feature drop

===== TEST =====
Original data.x shape: torch.Size([22500, 5])
Before drop - Has NaN: True
Before drop - Has Inf: False
After drop shape: torch.Size([22500, 3])
After drop - Has NaN: False
After drop - Has Inf: False
⚠️ WARNING: Invalid values already exist BEFORE feature drop


## 小图构造

In [10]:
# ===================== 0️⃣ 依赖 =====================
import torch
from torch.utils.data import Dataset, DataLoader

# ===================== 1️⃣ NodeDataset（不区分标签，只打包节点） =====================
class NodeDataset(Dataset):
    """
    Node-level Dataset（统一打包）：
    - 不区分有/无标签
    - mask 保留在 data 中，训练阶段再使用
    """
    def __init__(self, data):
        self.data = data
        self.node_idx = torch.arange(data.num_nodes)
        print(f"✔ Using ALL nodes, num_nodes = {data.num_nodes}")

    def __len__(self):
        return self.node_idx.size(0)

    def __getitem__(self, idx):
        return self.node_idx[idx]


# ===================== 2️⃣ Dataset 构建 =====================
train_dataset = NodeDataset(train_graph)
val_dataset   = NodeDataset(val_graph)
test_dataset  = NodeDataset(test_graph)


# ===================== 3️⃣ DataLoader（load 打包，不拆 mask） =====================
batch_size = 512*4   # ⭐ 4GB 显存建议 256 / 512

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)


# ===================== 4️⃣ 快速自检（确认打包正确） =====================
print("\n===== Loader Sanity Check =====")

batch_nodes = next(iter(train_loader))
print("[TRAIN]")
print("Batch size:", batch_nodes.size(0))
print("Min node id:", batch_nodes.min().item())
print("Max node id:", batch_nodes.max().item())

batch_nodes = next(iter(val_loader))
print("\n[VAL]")
print("Batch size:", batch_nodes.size(0))
print("Min node id:", batch_nodes.min().item())
print("Max node id:", batch_nodes.max().item())

batch_nodes = next(iter(test_loader))
print("\n[TEST]")
print("Batch size:", batch_nodes.size(0))
print("Min node id:", batch_nodes.min().item())
print("Max node id:", batch_nodes.max().item())

print("\n✅ Unified Node-level DataLoader setup finished.")


✔ Using ALL nodes, num_nodes = 45000
✔ Using ALL nodes, num_nodes = 7500
✔ Using ALL nodes, num_nodes = 22500

===== Loader Sanity Check =====
[TRAIN]
Batch size: 2048
Min node id: 8
Max node id: 44995

[VAL]
Batch size: 2048
Min node id: 0
Max node id: 2047

[TEST]
Batch size: 2048
Min node id: 0
Max node id: 2047

✅ Unified Node-level DataLoader setup finished.


# 模型初始化

In [11]:
import sys
import os

# ===== 添加项目根目录到 PYTHONPATH =====
PROJECT_ROOT = "/home/charles/HZU/Industrial_Software_Testing/Industrial_Software_Testing/High_similar_ML/HS_ML_v1"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("✔ Project root added to sys.path")


import torch
from src.model import GraphContrastiveLearner

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===== 从图中读取真实维度 =====
data = train_graph   # 你前面 torch.load 读出来的
in_dim = data.x.size(1)

# ===== 模型超参数=====
hidden_dim = 256
out_dim    = 128
proj_dim   = 128
tau        = 0.5

# ===== 初始化对比学习模型 =====
model = GraphContrastiveLearner(
    in_dim=in_dim,
    hidden_dim=hidden_dim,
    out_dim=out_dim,
    proj_dim=proj_dim,
    tau=tau
).to(device)

print(model)


✔ Project root added to sys.path
GraphContrastiveLearner(
  (encoder): GCNEncoder(
    (conv1): SAGEConv(3, 256, aggr=mean)
    (conv2): SAGEConv(256, 128, aggr=mean)
  )
  (projector): MLPHead(
    (fc1): Linear(in_features=128, out_features=128, bias=True)
    (ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (act): GELU(approximate='none')
    (fc2): Linear(in_features=128, out_features=128, bias=True)
  )
)


# 上游GCL训练

In [12]:
from torch_geometric.utils import subgraph
from torch_geometric.data import Data
# ===== 添加项目根目录到 PYTHONPATH =====
PROJECT_ROOT = "/home/charles/HZU/Industrial_Software_Testing/Industrial_Software_Testing/High_similar_ML/HS_ML_v1"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("✔ Project root added to sys.path")
from src.model import augment_graph

def train_one_epoch_with_loader(
    model,
    full_graph,
    loader,
    optimizer,
    feature_drop=0.1,
    edge_drop=0.1,
    noise_std=0.01
):
    model.train()
    device = next(model.parameters()).device

    total_loss = 0.0
    num_batches = 0

    for batch_nodes in loader:
        # ===================== 1️⃣ 子图构建（CPU） =====================
        batch_nodes_cpu = batch_nodes.cpu()

        edge_index, _ = subgraph(
            batch_nodes_cpu,
            full_graph.edge_index,     # CPU
            relabel_nodes=True,
            num_nodes=full_graph.num_nodes
        )

        x = full_graph.x[batch_nodes_cpu]

        # ===================== 2️⃣ 搬到 GPU =====================
        sub_data = Data(
            x=x.to(device),
            edge_index=edge_index.to(device)
        )

        # ===================== 3️⃣ 两个增强视图 =====================
        view1 = augment_graph(
            sub_data,
            feature_drop_prob=feature_drop,
            edge_drop_prob=edge_drop,
            noise_std=noise_std
        )

        view2 = augment_graph(
            sub_data,
            feature_drop_prob=feature_drop,
            edge_drop_prob=edge_drop,
            noise_std=noise_std
        )

        # ===================== 4️⃣ 对比损失 =====================
        loss = model.compute_loss(
            view1.x, view1.edge_index,
            view2.x, view2.edge_index
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / max(num_batches, 1)


✔ Project root added to sys.path


In [13]:
import torch

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

num_epochs = 200
log_interval = 10

for epoch in range(1, num_epochs + 1):
    loss = train_one_epoch_with_loader(
        model=model,
        full_graph=train_graph,   # ⭐ 整图
        loader=train_loader,      # ⭐ 统一 load
        optimizer=optimizer,
        feature_drop=0.1,
        edge_drop=0.1,
        noise_std=0.01
    )

    if epoch == 1 or epoch % log_interval == 0:
        print(f"[Epoch {epoch:03d}] Contrastive Loss = {loss:.4f}")

print("✅ Contrastive training (loader-based) finished.")


[Epoch 001] Contrastive Loss = 32.7604
[Epoch 010] Contrastive Loss = 31.0240
[Epoch 020] Contrastive Loss = 30.4277
[Epoch 030] Contrastive Loss = 29.9394
[Epoch 040] Contrastive Loss = 29.7559
[Epoch 050] Contrastive Loss = 29.6110
[Epoch 060] Contrastive Loss = 29.5011
[Epoch 070] Contrastive Loss = 29.4868
[Epoch 080] Contrastive Loss = 29.0455
[Epoch 090] Contrastive Loss = 28.8687
[Epoch 100] Contrastive Loss = 28.6703
[Epoch 110] Contrastive Loss = 28.6416
[Epoch 120] Contrastive Loss = 28.5373
[Epoch 130] Contrastive Loss = 28.5503
[Epoch 140] Contrastive Loss = 28.5072
[Epoch 150] Contrastive Loss = 28.5156
[Epoch 160] Contrastive Loss = 28.3918
[Epoch 170] Contrastive Loss = 28.3898
[Epoch 180] Contrastive Loss = 28.3334
[Epoch 190] Contrastive Loss = 28.3167
[Epoch 200] Contrastive Loss = 28.3698
✅ Contrastive training (loader-based) finished.


# 下游训练